# Evaluation of the models of `munich_new_site`

**Capability.** This scenario shows that the model predicts the cells of a site with no MDT record, as for a new site of a network plan.

**Purpose.** This notebook compares the U-Net with three baselines on the test records and,
for a synthetic scenario, on the true path gain map.

**Models.**

| Model | Path gain map |
|---|---|
| Sionna RT | The radio map of the original scene, with no change |
| Sionna RT + offset | Sionna RT plus one constant that the train records give (`fit_offset`) |
| SPM (Atoll form) | The form of the Standard Propagation Model of Atoll, with coefficients from a least-squares fit to the train records (`SpmModel`). This is the calibration that a planning tool does. |
| U-Net pretrained | The U-Net after the pretraining only |
| U-Net fine-tuned | The U-Net after the fine-tuning on the train records |

**Method.** Each model gives one path gain map in dB for each cell. The RSRP of a record is
the reference-signal power plus the path gain of the grid cell that holds its reported
position, for all models. The error of a pair is the measured RSRP minus the predicted RSRP.
`error_statistics` gives the statistics over the received pairs of the test part. For a
synthetic scenario, `map_statistics` compares each map with the true map over the grid cells
with a true path gain. The quantiles are `evaluation.quantiles`.

**Requirements.** Run `04_fine_tuning.ipynb` first.

## 1. Setup

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from experiments import (
    SPM_ATOLL,
    baseline_maps,
    cell_table,
    error_cdf_figure,
    error_map_figure,
    evaluated_cells,
    load_model,
    load_scenario,
    map_figure,
    map_rsrp_dbm,
    map_table,
    mdt_table,
)
from radio_calibration.evaluation import ResourceMonitor
from utils import load_config, save_figure, save_table, setup_notebook

cfg = load_config("scenario=munich_new_site")
device = setup_notebook(cfg)
data = load_scenario(cfg)

## 2. Maps of the models

In [ ]:
maps_db = baseline_maps(cfg, data) | {
    "U-Net pretrained": load_model(data, "pretrained", device).predict_path_gain_db(
        data.maps.target
    ),
}
with ResourceMonitor() as monitor:
    maps_db["U-Net fine-tuned"] = load_model(data, "fine_tuned", device).predict_path_gain_db(
        data.maps.target
    )

## 3. Error on the test records

Table 1 gives the statistics of each model on the test records. Table 2 gives the RMSE of each
model for each cell. Fig. 1 shows the distribution of the absolute error.

In [ ]:
display(
    save_table(
        mdt_table(cfg, data, maps_db)
        .style.set_caption("Table 1. Error on the test records.")
        .format(precision=3),
        "evaluation_mdt",
    )
)
save_table(
    cell_table(cfg, data, maps_db)
    .style.set_caption("Table 2. RMSE on the test records for each cell (dB).")
    .format(precision=2, na_rep="-"),
    "evaluation_by_cell",
)

In [ ]:
_, measured = data.arrays["test"]
errors = {
    model_name: measured - map_rsrp_dbm(data, path_gain_db, "test")
    for model_name, path_gain_db in maps_db.items()
}
fig = error_cdf_figure(errors)
save_figure(fig, "evaluation_error_cdf")
plt.show()

**Fig. 1.** Empirical distribution of the absolute error of each model on the test records.

## 4. Error against the true map

A synthetic scenario has the true path gain map of each cell. Table 3 compares each map with
it. The statistics use the evaluated cells: the test cells for the split method `site`, else
all cells. Fig. 2 shows the maps of one evaluated cell, and Fig. 3 shows the errors.

In [ ]:
truth = map_table(cfg, data, maps_db)
if truth is None:
    print("The scenario has no true map.")
else:
    display(
        save_table(
            truth.style.set_caption("Table 3. Error against the true map.").format(precision=3),
            "evaluation_true_map",
        )
    )

In [ ]:
shown_cell = int(evaluated_cells(cfg, data)[0])
print(f"Cell of Fig. 2 and Fig. 3: {data.cells[shown_cell].name}")
fig = map_figure(data, maps_db, shown_cell)
save_figure(fig, "evaluation_maps")
plt.show()

**Fig. 2.** Path gain maps of one evaluated cell. The first panel is the true map. The other
panels are the maps of the models, with the same color scale. The triangle is the antenna.
The text above the figure gives the cell.

In [ ]:
if data.maps.true_path_gain is not None:
    fig = error_map_figure(data, maps_db, shown_cell)
    save_figure(fig, "evaluation_error_maps")
    plt.show()

**Fig. 3.** True path gain minus the path gain of each model for the cell of Fig. 2. Red shows
that the model gives too little path gain, and blue too much. The color scale stops at the
95th percentile of the absolute error.

## 5. Comparison with the planning-tool calibration

The criterion of the scenario is that the fine-tuned U-Net has a smaller RMSE and a smaller
mean absolute error than the SPM model with the form of Atoll on the test records, and also on
the true map for a synthetic scenario. Table 5 gives the result. The notebook reports the
result also when the criterion is not met.

In [ ]:
mdt = mdt_table(cfg, data, maps_db)
checks = {
    "Test records, RMSE": (mdt.loc["U-Net fine-tuned", "rmse_db"], mdt.loc[SPM_ATOLL, "rmse_db"]),
    "Test records, MAE": (mdt.loc["U-Net fine-tuned", "mae_db"], mdt.loc[SPM_ATOLL, "mae_db"]),
}
if truth is not None:
    checks |= {
        "True map, RMSE": (
            truth.loc["U-Net fine-tuned", "rmse_db"],
            truth.loc[SPM_ATOLL, "rmse_db"],
        ),
        "True map, MAE": (truth.loc["U-Net fine-tuned", "mae_db"], truth.loc[SPM_ATOLL, "mae_db"]),
    }
criterion = pd.DataFrame(checks, index=["U-Net fine-tuned (dB)", "SPM (Atoll form) (dB)"]).T
criterion["U-Net is better"] = criterion.iloc[:, 0] < criterion.iloc[:, 1]
save_table(
    criterion.style.set_caption(
        "Table 5. U-Net against the SPM model with the form of Atoll."
    ).format(precision=3, subset=criterion.columns[:2]),
    "evaluation_criterion",
)

In [ ]:
save_table(
    pd.Series(monitor.summary())
    .to_frame("Value")
    .style.set_caption(
        "Table 6. Resource use of one prediction of all maps of the fine-tuned U-Net."
    )
    .format(precision=2),
    "evaluation_resources",
)